In [1]:
import os
import getpass

# Load the Hugging Face token by prompting the user
hf_token = getpass.getpass("Por favor, insira seu Hugging Face token: ")

# Set the HF_TOKEN environment variable
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("Hugging Face token loaded and set as environment variable.")
else:
    print("Nenhum token foi inserido.")

Por favor, insira seu Hugging Face token: ··········
Hugging Face token loaded and set as environment variable.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado com sucesso em '/content/drive'")

Mounted at /content/drive
Google Drive montado com sucesso em '/content/drive'


In [3]:
import os
import io
import json
import time
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import csv

proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained(
 "Salesforce/blip-image-captioning-base").to("cuda")

# Caminho para a pasta de imagens no Google Drive
image_directory = '/content/drive/MyDrive/Colab Notebooks/dados/train'

# Lista de extensões de imagem que serão consideradas
image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp')

# Obter todos os arquivos na pasta e filtrar por extensão de imagem
image_files = []
try:
    for filename in os.listdir(image_directory):
        if filename.lower().endswith(image_extensions):
            image_files.append(filename)
    print(f"Encontradas {len(image_files)} imagens na pasta: {image_directory}")
except FileNotFoundError:
    print(f"Erro: A pasta '{image_directory}' não foi encontrada. Certifique-se de que o Google Drive está montado e o caminho está correto.")
    image_files = [] # Garante que a lista esteja vazia se a pasta não for encontrada

if not image_files:
    print("Nenhuma imagem encontrada para processamento.")
else:
    print("Iniciando o processamento das imagens locais...")
    # Abrir o arquivo legendas.txt no modo de escrita dentro do image_directory
    with open(os.path.join(image_directory, 'legendas.txt'), 'w', encoding='utf-8') as f:
        for filename in image_files:
            full_image_path = os.path.join(image_directory, filename)
            try:
                img = Image.open(full_image_path).convert('RGB')
                saida = blip.generate(**proc(img, return_tensors='pt').to('cuda'), max_new_tokens=40)
                caption = proc.decode(saida[0], skip_special_tokens=True)
                print(f'{filename}: {caption}') # Mantém o print para feedback imediato
                f.write(f'{filename}\t estilo_cordel, {caption}\n') # Escreve no arquivo
            except Exception as e:
                print(f"Erro ao processar a imagem {filename}: {e}")
    print(f"Processamento concluído. Legendas salvas em '{os.path.join(image_directory, 'legendas.txt')}'.")
# Revisar manualmente e prefixar com o token do estilo antes de salvar

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Encontradas 11 imagens na pasta: /content/drive/MyDrive/Colab Notebooks/dados/train
Iniciando o processamento das imagens locais...
13981855656_7c7dbac94f_c.jpg: an old photo of a staircase in a house
lambe_lambe_unicamp_0861.jpg: a man is painting a picture of a man
lambe_lambe_pantoja.jpg: a man standing next to a red fire hydra
lambe_lambe_ultimo.jpg: a man standing next to a street sign
lambe_lambe_hayasaka.jpg: a man in an orange shirt
tintype_dama_idosa.jpg: an old photo of a woman in a dress
tintype_menino_boutonniere.jpg: a small portrait of a young boy
tintype_soldado_kepi.jpg: a small portrait of a man in a red frame
tintype_familia_cinco.jpg: a group of children posing for a picture
tintype_jovem_joias.jpg: a black man in a suit and tie sitting on a chair
tintype_menino_flor.jpg: a black man in a suit and tie with a bow tie
Processamento concluído. Legendas salvas em '/content/drive/MyDrive/Colab Notebooks/dados/train/legendas.txt'.


In [4]:
path_metadata = os.path.join(image_directory, 'metadata.jsonl')

# Abre o arquivo metadata.jsonl em modo de escrita
with open(path_metadata, 'w', encoding='utf-8') as f_jsonl:
    # Lista todos os arquivos da pasta
    arquivos = os.listdir(image_directory)

    count = 0
    for arquivo in arquivos:
        # Verifica se o arquivo é uma imagem
        if arquivo.lower().endswith(image_extensions):
            nome_base, _ = os.path.splitext(arquivo)
            arquivo_txt = nome_base + '.txt'
            caminho_txt = os.path.join(image_directory, arquivo_txt)

            # Verifica se o arquivo .txt correspondente existe
            if os.path.exists(caminho_txt):
                # Lê a legenda de dentro do arquivo .txt
                with open(caminho_txt, 'r', encoding='utf-8') as f_txt:
                    legenda = f_txt.read().strip()

                # Cria a estrutura do dicionário (JSON)
                item = {
                    "file_name": arquivo,
                    "text": legenda
                }

                # Converte em string JSON e salva uma linha no arquivo .jsonl
                f_jsonl.write(json.dumps(item, ensure_ascii=False) + '\n')
                count += 1